In [ ]:
import marimo as mo

In [ ]:
import sys
import os
import subprocess

# Do not import torch here. The sandbox dependency environment must be
# established first; the H2/ES1 runner imports and validates torch itself.

In [ ]:
os.environ["HF_HUB_DISABLE_XET"] = "1"

from pathlib import Path as _VarHfPath
import zipfile as _var_hf_zipfile
from huggingface_hub import hf_hub_download

# HF_TOKEN is a runtime secret; never embed a personal access token in notebook source.
HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
os.environ["HF_TOKEN"] = HF_TOKEN

VARROA_HF_REPO = "TLMHoang/varroa_data"
VARROA_HF_FILENAME = "Archive.zip"
varroa_download_dir = _VarHfPath("/marimo/varroa_download")
varroa_download_dir.mkdir(parents=True, exist_ok=True)

_expected_archive = varroa_download_dir / VARROA_HF_FILENAME
try:
    varroa_archive_path = _VarHfPath(
        hf_hub_download(
            repo_id=VARROA_HF_REPO,
            filename=VARROA_HF_FILENAME,
            repo_type="dataset",
            token=HF_TOKEN,
            local_dir=str(varroa_download_dir),
            force_download=False,
        )
    ).resolve()
except Exception:
    if _expected_archive.is_file() and _var_hf_zipfile.is_zipfile(_expected_archive):
        varroa_archive_path = _expected_archive.resolve()
        print("Hugging Face metadata request failed; reusing verified local ZIP.")
    else:
        raise

if not varroa_archive_path.is_file():
    raise FileNotFoundError(varroa_archive_path)
if not _var_hf_zipfile.is_zipfile(varroa_archive_path):
    raise ValueError(f"Downloaded file is not a valid ZIP: {varroa_archive_path}")

print(
    "Varroa ZIP ready:",
    varroa_archive_path,
    f"({varroa_archive_path.stat().st_size / 1024**3:.2f} GiB)",
)

In [ ]:
# ================================================================
# EXTRACT THE COMPLETE VARROA ARCHIVE — run after HF download.
# Resolves a wrapper directory safely and validates the native source layout.
# ================================================================
from pathlib import Path as _VarPath, PurePosixPath as _VarPurePosixPath
import json as _var_json
import zipfile as _var_zipfile

VARROA_SOURCE_SPLITS = ("train", "val", "test")
varroa_extract_dir = _VarPath("/marimo/varroa_extracted").resolve()


def _is_varroa_source_root(_root: _VarPath) -> bool:
    return all(
        (_root / _split / "videos").is_dir()
        and (_root / _split / "gt_one.csv").is_file()
        for _split in VARROA_SOURCE_SPLITS
    )


def _resolve_varroa_root(_extract_dir: _VarPath) -> _VarPath:
    _candidates = set()
    if _is_varroa_source_root(_extract_dir):
        _candidates.add(_extract_dir.resolve())
    for _gt_file in _extract_dir.rglob("gt_one.csv"):
        _candidate = _gt_file.parent.parent
        if _is_varroa_source_root(_candidate):
            _candidates.add(_candidate.resolve())
    _valid_roots = sorted(_candidates)
    if len(_valid_roots) != 1:
        raise RuntimeError(
            "Expected exactly one Varroa root containing train/val/test videos + gt_one.csv, found: "
            f"{[str(_root) for _root in _valid_roots]}"
        )
    return _valid_roots[0]


if not _var_zipfile.is_zipfile(varroa_archive_path):
    raise ValueError(f"Invalid Varroa ZIP: {varroa_archive_path}")

varroa_extract_dir.mkdir(parents=True, exist_ok=True)
_extract_marker = varroa_extract_dir / ".varroa_extract_complete.json"
_archive_stat = varroa_archive_path.stat()
_archive_signature = {
    "archive": str(varroa_archive_path),
    "size": int(_archive_stat.st_size),
    "mtime_ns": int(_archive_stat.st_mtime_ns),
}
_marker_matches = False
if _extract_marker.is_file():
    try:
        _marker_matches = _var_json.loads(
            _extract_marker.read_text(encoding="utf-8")
        ) == _archive_signature
    except (OSError, ValueError):
        _marker_matches = False

if _marker_matches:
    print("SKIP Varroa extraction: completed archive signature matches.")
else:
    print("Extracting the complete Varroa archive...")
    with _var_zipfile.ZipFile(varroa_archive_path) as _archive:
        _unsafe_members = []
        for _member in _archive.infolist():
            _member_path = _VarPurePosixPath(_member.filename)
            if _member_path.is_absolute() or ".." in _member_path.parts:
                _unsafe_members.append(_member.filename)
        if _unsafe_members:
            raise ValueError(
                f"Refusing unsafe paths in Varroa ZIP: {_unsafe_members[:5]}"
            )
        _archive.extractall(varroa_extract_dir)
    _extract_marker.write_text(
        _var_json.dumps(_archive_signature, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    print("Varroa archive extraction complete.")

varroa_data_root = _resolve_varroa_root(varroa_extract_dir)
print("Resolved Varroa data root:", varroa_data_root)
for _split in VARROA_SOURCE_SPLITS:
    print(
        f"  {_split}:",
        varroa_data_root / _split / "videos",
        varroa_data_root / _split / "gt_one.csv",
    )

In [ ]:
# Runtime packages required by the patched local Ultralytics and Varroa pipeline.
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "albumentations",
        "kornia",
        "timm",
        "huggingface_hub",
        "opencv-python-headless",
        "pillow",
        "pyyaml",
        "ultralytics-thop",
    ],
    check=True,
)

In [ ]:
# ================================================================
# GITHUB TOKEN — live-only cell.
# Dán token vào đây nếu repo GitHub private hoặc cần tránh auth prompt.
# Không push/export notebook public khi cell này chứa token thật.
# ================================================================
import os as _yolo_token_os

# TODO: paste token here in live if needed, e.g. "github_pat_...".
yolo_github_token = "[REDACTED_SECRET]"

if yolo_github_token:
    _yolo_token_os.environ["GITHUB_TOKEN"] = yolo_github_token

print("GITHUB_TOKEN configured for GitHub clone/pull:", bool(_yolo_token_os.environ.get("GITHUB_TOKEN")))

In [ ]:
# ================================================================
# LOAD THE MAIN YOLO_FTSC CODEBASE FROM GITHUB — full repo sync.
# Varroa source data is downloaded/extracted by the cells above.
# ================================================================
from pathlib import Path as _YoloCodePath
import os as _yolo_code_os
import shutil as _yolo_code_shutil
import subprocess as _yolo_code_subprocess
import time as _yolo_code_time
from urllib.parse import urlparse as _yolo_urlparse, urlunparse as _yolo_urlunparse


yolo_github_repo_url = "https://github.com/TLMHoang-AI/YOLO_FTSC.git"
yolo_code_dir = _YoloCodePath("/marimo/yolo_code")
yolo_github_branch = "main"


def _yolo_tokenized_url(_url: str) -> str:
    _token = yolo_github_token or _yolo_code_os.environ.get("GITHUB_TOKEN", "")
    if not _token or not _url.startswith("https://github.com/"):
        return _url
    _parsed = _yolo_urlparse(_url)
    return _yolo_urlunparse(
        _parsed._replace(netloc=f"x-access-token:{_token}@{_parsed.netloc}")
    )


def _yolo_redact(_parts):
    _token = yolo_github_token or _yolo_code_os.environ.get("GITHUB_TOKEN", "")
    if not _token:
        return _parts
    return [_part.replace(_token, "<GITHUB_TOKEN>") for _part in _parts]


def _yolo_git(_args, *, _cwd=None, _check=True):
    _env = dict(_yolo_code_os.environ)
    _env["GIT_TERMINAL_PROMPT"] = "0"
    _result = _yolo_code_subprocess.run(
        ["git", *_args], cwd=_cwd, env=_env, text=True, capture_output=True
    )
    if _result.stdout:
        print(_result.stdout)
    if _result.stderr:
        print(_result.stderr)
    if _check and _result.returncode != 0:
        _safe = " ".join(_yolo_redact(["git", *_args]))
        raise RuntimeError(f"Git command failed: {_safe}")
    return _result


_yolo_core_required_markers = [
    yolo_code_dir / "misc/prepare_dataset.py",
    yolo_code_dir / "train_levir_scripts/train_varroa_dbss_hit.py",
    yolo_code_dir / "train_levir_scripts/train_varroa_ftsc_next_ablation_suite.py",
    yolo_code_dir / "utils/hf_result_archive.py",
    yolo_code_dir / "utils/hf_checkpoint_recovery.py",
    yolo_code_dir / "utils/ftsc_completion_planner.py",
    yolo_code_dir / "train_levir_scripts/complete_varroa_ftsc_fullmetric_v1.py",
    yolo_code_dir / "evaluate_test/standard_detection_metrics.py",
    yolo_code_dir / "evaluate_test/size_bucket_evaluator.py",
    yolo_code_dir / "models_related/models_config/yolov8/varroa/yolov8n_varroa_ftsc_v1_p2p3.yaml",
    yolo_code_dir / "models_related/models_config/yolov8/varroa/yolov8n_varroa_ftsc_v2_p2p3_edge025.yaml",
    yolo_code_dir / "models_related/models_config/yolov8/varroa/yolov8n_varroa_ftsc_edge_p2_detect_only.yaml",
    yolo_code_dir / "models_related/models_config/yolov8/varroa/yolov8n_varroa_ftsc_edge_p3_detect_only.yaml",
    yolo_code_dir / "models_related/models_config/yolov8/varroa/yolov8n_varroa_ftsc_head_p234.yaml",
    yolo_code_dir / "models_related/models_config/yolov8/varroa/yolov8n_varroa_ftsc_p234_edge_p2_detect_only.yaml",
    yolo_code_dir / "models_related/models_config/yolov8/varroa/yolov8n_varroa_ftsc_head_p234_edge_p3.yaml",
    yolo_code_dir / "models_related/models_config/yolov8/varroa/yolov8n_varroa_ftsc_head_p34.yaml",
    yolo_code_dir / "models_related/models_config/yolov8/varroa/yolov8n_varroa_ftsc_head_p34_edge_p3.yaml",
    yolo_code_dir / "models_related/ultralytics/ultralytics/engine/trainer.py",
    yolo_code_dir / "models_related/ultralytics/ultralytics/nn/modules/ftsc.py",
    yolo_code_dir / "models_related/ultralytics/ultralytics/nn/modules/edge_cue.py",
    yolo_code_dir / "models_related/ultralytics/ultralytics/nn/modules/color_cue.py",
    yolo_code_dir / "models_related/ultralytics/ultralytics/utils/loss.py",
]


def _yolo_missing_markers():
    return [str(_path) for _path in _yolo_core_required_markers if not _path.exists()]


_repo_url_for_clone = _yolo_tokenized_url(yolo_github_repo_url)
if (yolo_code_dir / ".git").is_dir():
    print("Existing yolo_code git repo found; forcing full GitHub sync...")
    _yolo_git(["remote", "set-url", "origin", _repo_url_for_clone], _cwd=yolo_code_dir)
    _yolo_git(["fetch", "origin", yolo_github_branch], _cwd=yolo_code_dir)
    _yolo_git(["reset", "--hard", f"origin/{yolo_github_branch}"], _cwd=yolo_code_dir)
    _yolo_git(["clean", "-fd", "-e", "runs/", "-e", "datasets/", "-e", "yolov8n.pt", "-e", "yolov9t.pt", "-e", "yolov10n.pt", "-e", "yolo11n.pt"], _cwd=yolo_code_dir)
    _yolo_git(["remote", "set-url", "origin", yolo_github_repo_url], _cwd=yolo_code_dir)
elif yolo_code_dir.exists():
    _backup = yolo_code_dir.with_name(
        f"{yolo_code_dir.name}_partial_backup_{int(_yolo_code_time.time())}"
    )
    print(f"Found non-git/partial yolo_code. Moving it to backup: {_backup}")
    _yolo_code_shutil.move(str(yolo_code_dir), str(_backup))
    print("Cloning FULL YOLO_FTSC repo from GitHub...")
    _yolo_git(["clone", "--branch", yolo_github_branch, _repo_url_for_clone, str(yolo_code_dir)])
    _yolo_git(["remote", "set-url", "origin", yolo_github_repo_url], _cwd=yolo_code_dir)
else:
    print("Cloning FULL YOLO_FTSC repo from GitHub...")
    _yolo_git(["clone", "--branch", yolo_github_branch, _repo_url_for_clone, str(yolo_code_dir)])
    _yolo_git(["remote", "set-url", "origin", yolo_github_repo_url], _cwd=yolo_code_dir)

_missing_yolo = _yolo_missing_markers()
if _missing_yolo:
    raise FileNotFoundError(
        "YOLO FTSC sync is missing required Varroa next-ablation files:\n- "
        + "\n- ".join(_missing_yolo)
    )

print("Main YOLO_FTSC codebase ready:", yolo_code_dir)
print("Latest commit:")
_yolo_git(["log", "--oneline", "-1"], _cwd=yolo_code_dir)

In [ ]:
# ================================================================
# LOAD CANONICAL VARROA PREPARATION FROM THE FRESH yolo_code.
# The deferred next-ablation execution phase materializes requested seeds through dataset_for_seed().
# This cell validates the canonical API/contract without materializing all seeds.
# ================================================================
from pathlib import Path as _VarPrepPath
import importlib.util as _var_prepare_importlib_util
import sys as _var_prepare_sys

VARROA_PREPARE_SEEDS = (42, 43, 44)
varroa_legacy_runner_file = (
    _VarPrepPath(yolo_code_dir)
    / "train_levir_scripts/train_varroa_dbss_hit.py"
).resolve()
varroa_prepared_root = _VarPrepPath("/marimo/datasets/varroa_ftsc").resolve()

if not varroa_legacy_runner_file.is_file():
    raise FileNotFoundError(varroa_legacy_runner_file)

_yolo_root_for_varroa = _VarPrepPath(yolo_code_dir).resolve()
_varroa_script_dir = varroa_legacy_runner_file.parent
for _path in (_yolo_root_for_varroa, _varroa_script_dir):
    if str(_path) not in _var_prepare_sys.path:
        _var_prepare_sys.path.insert(0, str(_path))

_var_prepare_module_name = "_live_main_yolo_varroa_prepare"
_var_prepare_sys.modules.pop(_var_prepare_module_name, None)
_var_prepare_spec = _var_prepare_importlib_util.spec_from_file_location(
    _var_prepare_module_name,
    varroa_legacy_runner_file,
)
if _var_prepare_spec is None or _var_prepare_spec.loader is None:
    raise ImportError(f"Cannot import Varroa runner: {varroa_legacy_runner_file}")
varroa_legacy_module = _var_prepare_importlib_util.module_from_spec(
    _var_prepare_spec
)
_var_prepare_sys.modules[_var_prepare_module_name] = varroa_legacy_module
_var_prepare_spec.loader.exec_module(varroa_legacy_module)

for _required_function in (
    "dataset_for_seed",
    "validate_dataset",
    "seed_everything",
    "train_kwargs",
    "evaluate",
):
    if not callable(getattr(varroa_legacy_module, _required_function, None)):
        raise RuntimeError(f"Pulled Varroa runner is missing {_required_function}()")

if varroa_legacy_module.EXPECTED_SPLITS != {
    "train": 2762,
    "val": 592,
    "test": 592,
}:
    raise RuntimeError(
        f"Varroa expected counts drifted: {varroa_legacy_module.EXPECTED_SPLITS}"
    )

varroa_prepared_root.mkdir(parents=True, exist_ok=True)
print("Varroa canonical preparation API ready:")
print("  source root:", varroa_data_root)
print("  prepared root:", varroa_prepared_root)
print("  canonical seeds:", VARROA_PREPARE_SEEDS)
print("  seed materialization deferred until HF-first Varroa evaluation")

In [ ]:
# Varroa 17 historical cases: 4 causal, 4 Edge/head-scale, 9 next-ablation.
# Evaluation-only HF recovery. Alpha4/Alpha8 are NOT historical.
VARROA_CAUSAL_CASES = (
    "V0_NOFTSC", "V1_FTSC", "V2_EDGE025", "V3_COLOR",
)
VARROA_HEADSCALE_CASES = (
    "EDGE_P2_DETECT_ONLY", "EDGE_P3_DETECT_ONLY",
    "HEAD_P234", "HEAD_P234_EDGE_P3",
)
VARROA_NEXT_ABLATION_CASES = (
    "FTSC_P234_EDGE_P2_DETECT_ONLY", "FTSC_P34", "FTSC_P34_EDGE_P3_DETECT_ONLY",
    "FTSC_P23_NOMOSAIC", "FTSC_P23_CP2", "FTSC_P23_NOMOSAIC_CP2",
    "EDGE_P2_DETECT_ONLY_NOMOSAIC", "EDGE_P2_DETECT_ONLY_CP2",
    "EDGE_P2_DETECT_ONLY_NOMOSAIC_CP2",
)
VARROA_HISTORICAL_FULLMETRIC_CASES = (
    VARROA_CAUSAL_CASES + VARROA_HEADSCALE_CASES + VARROA_NEXT_ABLATION_CASES
)
if len(VARROA_HISTORICAL_FULLMETRIC_CASES) != 17 or len(set(VARROA_HISTORICAL_FULLMETRIC_CASES)) != 17:
    raise RuntimeError("Expected exactly 17 unique historical cases")
print("Historical Varroa registry: 17 cases / 51 seeds")
for _group, _cases in (
    ("causal", VARROA_CAUSAL_CASES),
    ("headscale", VARROA_HEADSCALE_CASES),
    ("next-ablation", VARROA_NEXT_ABLATION_CASES),
):
    print(_group, len(_cases), ", ".join(_cases))

## Varroa — HF-first six-metric recovery of 17 historical cases

This notebook backfills the existing 4 causal + 4 Edge/head-scale + 9 next-ablation cases, seeds 42/43/44. It does not launch new training. Use phases plan → restore → evaluate → aggregate → optional publish. Inspect the plan for V0/V1 before scaling to all cases. Never take TRAIN_REQUIRED as automatic authorization to train.

The Alpha4/Alpha8 training launcher is retained but disabled. Existing credential/key cells are preserved.

In [ ]:
# Legacy next-ablation training remains disabled; historical cases use HF-first metric recovery.
RUN_VARROA_FTSC_NEXT_ABLATION = False

In [ ]:
# Legacy Alpha training launcher preserved, but disabled for historical recovery.
RUN_VARROA_FTSC_ALPHA = False
VARROA_FTSC_ALPHA_PHASE = "preflight"  # legacy Alpha launcher disabled; do not train here
VARROA_FTSC_ALPHA_CASES = ("ALPHA4_rerun", "ALPHA8_rerun")
VARROA_FTSC_ALPHA_SEEDS = (42, 43, 44)
VARROA_FTSC_ALPHA_WORKERS = 16
VARROA_FTSC_ALPHA_DEVICE = "cuda"
VARROA_FTSC_ALPHA_LIVE_MODEL_PREFLIGHT = False
ARCHIVE_VARROA_FTSC_ALPHA = False
UPLOAD_VARROA_FTSC_ALPHA_AFTER_EACH_RUN = False
if RUN_VARROA_FTSC_ALPHA:
    from pathlib import Path as _VarAlphaPath
    import importlib.util as _var_alpha_importlib_util
    import sys as _var_alpha_sys
    _var_alpha_file = _VarAlphaPath(yolo_code_dir) / "train_levir_scripts/train_varroa_ftsc_alpha_sensitivity_v1.py"
    if not _var_alpha_file.is_file():
        raise FileNotFoundError(
            f"Missing synced alpha runner: {_var_alpha_file}. Push the local implementation, then rerun yolo_code_sync."
        )
    _var_alpha_module_name = "_live_varroa_ftsc_alpha_sensitivity_v1"
    _var_alpha_sys.modules.pop(_var_alpha_module_name, None)
    _var_alpha_spec = _var_alpha_importlib_util.spec_from_file_location(_var_alpha_module_name, _var_alpha_file)
    if _var_alpha_spec is None or _var_alpha_spec.loader is None:
        raise ImportError(f"Cannot load alpha runner: {_var_alpha_file}")
    _var_alpha = _var_alpha_importlib_util.module_from_spec(_var_alpha_spec)
    _var_alpha_sys.modules[_var_alpha_module_name] = _var_alpha
    _var_alpha_spec.loader.exec_module(_var_alpha)
    _var_alpha_cli = [
        "--phase", VARROA_FTSC_ALPHA_PHASE,
        "--data-root", str(varroa_data_root),
        "--dataset-root", str(varroa_prepared_root),
        "--project", str(_VarAlphaPath(yolo_code_dir) / "runs/varroa_ftsc_position_alpha_h2_v1"),
        "--pretrained", str(_VarAlphaPath(yolo_code_dir) / "yolov8n.pt"),
        "--workers", str(VARROA_FTSC_ALPHA_WORKERS),
        "--device", VARROA_FTSC_ALPHA_DEVICE,
        "--cases", *VARROA_FTSC_ALPHA_CASES,
        "--seeds", *[str(_seed) for _seed in VARROA_FTSC_ALPHA_SEEDS],
    ]
    if VARROA_FTSC_ALPHA_LIVE_MODEL_PREFLIGHT:
        _var_alpha_cli.append("--live-model-preflight")
    if VARROA_FTSC_ALPHA_PHASE != "preflight":
        from utils.hf_result_archive import HFResultArchive as _VarAlphaArchive
        _var_alpha_archive = _VarAlphaArchive(repo_id="TLMHoang/FTSC_results", token=HF_TOKEN, cache_dir="/marimo/hf_results_cache/varroa_ftsc_position_alpha_h2_v1")
        _var_alpha_archive.restore_completed_runs(dataset_slug=_var_alpha.NAMESPACE, project=_VarAlphaPath(yolo_code_dir) / "runs" / _var_alpha.NAMESPACE, variants=VARROA_FTSC_ALPHA_CASES, seeds=VARROA_FTSC_ALPHA_SEEDS, run_name_fn=lambda _case, _seed: f"seed_{_seed}", required_artifacts=(*_var_alpha.TRAIN_ARTIFACTS, "experiment_manifest.json", _var_alpha.METRICS_ARTIFACT, _var_alpha.COMPLETE_ARTIFACT))
    if VARROA_FTSC_ALPHA_PHASE == "train" and UPLOAD_VARROA_FTSC_ALPHA_AFTER_EACH_RUN:
        _var_alpha_case_index = _var_alpha_cli.index("--cases")
        _var_alpha_base_cli = _var_alpha_cli[:_var_alpha_case_index]
        for _case in VARROA_FTSC_ALPHA_CASES:
            for _seed in VARROA_FTSC_ALPHA_SEEDS:
                _run = _VarAlphaPath(yolo_code_dir) / "runs" / _var_alpha.NAMESPACE / _case / f"seed_{_seed}"
                _training_ready = all((_run / _relative).is_file() for _relative in _var_alpha.TRAIN_ARTIFACTS)
                if not _training_ready:
                    _var_alpha.main([*_var_alpha_base_cli, "--cases", _case, "--seeds", str(_seed)])
                _evaluate_cli = [*_var_alpha_base_cli, "--cases", _case, "--seeds", str(_seed)]
                _evaluate_cli[1] = "evaluate"
                if not (_run / _var_alpha.COMPLETE_ARTIFACT).is_file():
                    _var_alpha.main(_evaluate_cli)
                _validate_cli = [*_evaluate_cli]
                _validate_cli[1] = "validate"
                _var_alpha.main(_validate_cli)
                _var_alpha_archive.publish_run(
                    dataset_slug=_var_alpha.NAMESPACE,
                    variant=_case,
                    seed=_seed,
                    run_name=_run.name,
                    run_dir=_run,
                    required_artifacts=(*_var_alpha.TRAIN_ARTIFACTS, "experiment_manifest.json", _var_alpha.METRICS_ARTIFACT, _var_alpha.COMPLETE_ARTIFACT),
                )
                print(f"TRAIN+EVAL+VALIDATE+UPLOAD complete: {_case}/seed_{_seed}", flush=True)
    else:
        _var_alpha.main(_var_alpha_cli)
    if ARCHIVE_VARROA_FTSC_ALPHA and VARROA_FTSC_ALPHA_PHASE == "archive":
        for _case in VARROA_FTSC_ALPHA_CASES:
            for _seed in VARROA_FTSC_ALPHA_SEEDS:
                _run = _VarAlphaPath(yolo_code_dir) / "runs" / _var_alpha.NAMESPACE / _case / f"seed_{_seed}"
                _var_alpha_archive.publish_run(dataset_slug=_var_alpha.NAMESPACE, variant=_case, seed=_seed, run_name=_run.name, run_dir=_run, required_artifacts=(*_var_alpha.TRAIN_ARTIFACTS, "experiment_manifest.json", _var_alpha.METRICS_ARTIFACT, _var_alpha.COMPLETE_ARTIFACT))

In [ ]:
# HF-first completion for 17 historical Varroa cases (NO training).
# Phase sequence: plan -> restore -> evaluate -> aggregate -> publish (explicit).
RUN_VARROA_FTSC_FULLMETRIC_RECOVERY = False
VARROA_FTSC_FULLMETRIC_PHASE = "plan"
VARROA_FTSC_FULLMETRIC_CASES = VARROA_HISTORICAL_FULLMETRIC_CASES
VARROA_FTSC_FULLMETRIC_SEEDS = (42, 43, 44)
VARROA_FTSC_FULLMETRIC_DEVICE = "cuda"
VARROA_FTSC_FULLMETRIC_WORKERS = 8
VARROA_FTSC_FULLMETRIC_ALLOW_PUBLISH = False
if RUN_VARROA_FTSC_FULLMETRIC_RECOVERY:
    from pathlib import Path as _VarRecoverPath
    import subprocess as _var_recover_subprocess
    import os as _var_recover_os
    import json as _var_recover_json
    from collections import Counter as _VarRecoverCounter

    _cases = tuple(VARROA_FTSC_FULLMETRIC_CASES)
    _seeds = tuple(VARROA_FTSC_FULLMETRIC_SEEDS)
    _phase = VARROA_FTSC_FULLMETRIC_PHASE
    if (not _cases or len(_cases) != len(set(_cases))
            or not set(_cases).issubset(set(VARROA_HISTORICAL_FULLMETRIC_CASES))):
        raise ValueError("Only unique selected cases from the 17 historical Varroa cases are allowed")
    if (not _seeds or len(_seeds) != len(set(_seeds))
            or not set(_seeds).issubset({42, 43, 44})):
        raise ValueError("Allowed seeds are 42/43/44 without duplicates")
    if _phase not in {"plan", "restore", "evaluate", "aggregate", "publish"}:
        raise ValueError(f"Unsupported Varroa fullmetric phase: {_phase}")
    if _phase == "aggregate" and set(_seeds) != {42, 43, 44}:
        raise ValueError("Publication-quality aggregate requires seeds 42,43,44")
    if _phase == "publish" and not VARROA_FTSC_FULLMETRIC_ALLOW_PUBLISH:
        raise RuntimeError("Publishing requires VARROA_FTSC_FULLMETRIC_ALLOW_PUBLISH = True")

    _script = (_VarRecoverPath(yolo_code_dir) /
               "train_levir_scripts/complete_varroa_ftsc_fullmetric_v1.py")
    if not _script.is_file():
        raise FileNotFoundError(f"Missing synced runner: {_script}; push the local code then rerun yolo_code_sync")
    _project = _VarRecoverPath(yolo_code_dir) / "runs/varroa_ftsc_fullmetric_recovery_v1"
    _cmd = [
        "python", str(_script), "--phase", _phase,
        "--data-root", str(varroa_data_root),
        "--dataset-root", str(varroa_prepared_root),
        "--project", str(_project),
        "--cache-dir", "/marimo/hf_results_cache/varroa_fullmetric_recovery",
        "--device", VARROA_FTSC_FULLMETRIC_DEVICE,
        "--workers", str(VARROA_FTSC_FULLMETRIC_WORKERS),
        "--cases", *_cases,
        "--seeds", *(str(_seed) for _seed in _seeds),
    ]
    # Stream child stdout/stderr live to this Marimo cell (do not buffer 51 runs).
    from collections import deque as _VarRecoverDeque
    print(f"[VARROA] phase={_phase}, cases={len(_cases)}, seeds={_seeds}", flush=True)
    print(f"[VARROA] project={_project}", flush=True)
    _tail = _VarRecoverDeque(maxlen=80)
    _proc = _var_recover_subprocess.Popen(
        _cmd, cwd=str(yolo_code_dir),
        env={**_var_recover_os.environ, "HF_TOKEN": HF_TOKEN, "PYTHONUNBUFFERED": "1"},
        stdout=_var_recover_subprocess.PIPE,
        stderr=_var_recover_subprocess.STDOUT,
        text=True, bufsize=1,
    )
    if _proc.stdout is None:
        raise RuntimeError("Recovery subprocess stdout pipe unavailable")
    try:
        for _line in _proc.stdout:
            _safe_line = _line.rstrip("\n").replace(HF_TOKEN, "<HF_TOKEN>")
            _tail.append(_safe_line)
            print(_safe_line, flush=True)
    finally:
        _proc.stdout.close()
    _returncode = _proc.wait()
    if _returncode != 0:
        raise RuntimeError(
            f"Varroa fullmetric {_phase} failed (exit {_returncode}):\n"
            + "\n".join(_tail)
        )
    _report_file = {
        "plan": "recovery_plan.json",
        "restore": "recovery_plan.json",
        "evaluate": "evaluation_status.json",
    }.get(_phase)
    if _report_file:
        _report_path = _project / _report_file
        if not _report_path.is_file():
            raise FileNotFoundError(f"Phase {_phase} ended without {_report_path}")
        _report = _var_recover_json.loads(_report_path.read_text(encoding="utf-8"))
        _status_counts = _VarRecoverCounter(
            _row.get("action", _row.get("status", "UNKNOWN"))
            for _row in _report.get("rows", [])
        )
        print("Varroa", _phase, "decision counts:", dict(_status_counts))
        if _report.get("inventory_status") == "BLOCKED_HF_UNAVAILABLE":
            print("HF inventory unavailable; no train action is authorized.")
    else:
        print(f"[VARROA] {_phase} subprocess finished successfully.", flush=True)
    print("Recovery artifacts:", _project)
    print("Selection:", len(_cases), "cases x", len(_seeds), "seeds")

In [ ]:
# Download the canonical YOLOv8n pretrained checkpoint into the synced repository.
from pathlib import Path as _YoloWeightPath
import hashlib as _yolo_weight_hashlib
import urllib.request as _yolo_weight_urllib

_YOLOV8N_URL = "https://github.com/ultralytics/assets/releases/download/v8.4.0/yolov8n.pt"
_YOLOV8N_SHA256 = "f59b3d833e2ff32e194b5bb8e08d211dc7c5bdf144b90d2c8412c47ccfc83b36"
yolov8n_pretrained_path = (_YoloWeightPath(yolo_code_dir) / "yolov8n.pt").resolve()


def _yolov8n_sha256(_path: _YoloWeightPath) -> str:
    _digest = _yolo_weight_hashlib.sha256()
    with _path.open("rb") as _stream:
        for _block in iter(lambda: _stream.read(1024 * 1024), b""):
            _digest.update(_block)
    return _digest.hexdigest()


if yolov8n_pretrained_path.is_file():
    _actual_sha256 = _yolov8n_sha256(yolov8n_pretrained_path)
    if _actual_sha256 != _YOLOV8N_SHA256:
        raise RuntimeError(
            f"Refusing unexpected existing checkpoint {yolov8n_pretrained_path}: "
            f"sha256={_actual_sha256}"
        )
    print("SKIP YOLOv8n download: verified checkpoint already exists.")
else:
    yolov8n_pretrained_path.parent.mkdir(parents=True, exist_ok=True)
    _temporary_checkpoint = yolov8n_pretrained_path.with_suffix(".pt.download")
    try:
        _yolo_weight_urllib.urlretrieve(_YOLOV8N_URL, _temporary_checkpoint)
        _actual_sha256 = _yolov8n_sha256(_temporary_checkpoint)
        if _actual_sha256 != _YOLOV8N_SHA256:
            raise RuntimeError(
                f"Downloaded YOLOv8n checksum mismatch: sha256={_actual_sha256}"
            )
        _temporary_checkpoint.replace(yolov8n_pretrained_path)
    finally:
        if _temporary_checkpoint.exists():
            _temporary_checkpoint.unlink()
    print("Downloaded and verified canonical YOLOv8n checkpoint.")

print("YOLOv8n:", yolov8n_pretrained_path)
print("SHA256:", _yolov8n_sha256(yolov8n_pretrained_path))


In [ ]:
# Archived cross-YOLO launcher: disabled during FTSC journal Phase 2.
if False:
    # Overnight full pipeline: Git sync -> dataset -> preflight -> 18 runs -> HF -> aggregate.
    # Start the notebook once (Run All). This cell launches the entire suite automatically.
    # On a subsequent Run All, completed HF-backed runs are restored and verified;
    # partial last.pt runs resume without manually switching phases.
    from pathlib import Path as _VarCYPath
    import os as _VarCYOS
    import subprocess as _VarCYSubprocess
    import sys as _VarCYSys
    
    _VarCYRoot = _VarCYPath(yolo_code_dir).resolve()
    _VarCYScript = _VarCYRoot / "utils/ftsc_cross_yolo_overnight_v1.py"
    if not _VarCYScript.is_file():
        raise FileNotFoundError(
            f"Missing overnight runner in GitHub-synced codebase: {_VarCYScript}. "
            "Push the cross-YOLO codebase changes and rerun Git sync."
        )
    _VarCYCommand = [
        _VarCYSys.executable, "-u", str(_VarCYScript),
        "--dataset", "varroa",
        "--data-root", str(_VarCYPath(varroa_data_root).resolve()),
        "--dataset-root", str(_VarCYPath(varroa_prepared_root).resolve()),
        "--device", "cuda",
        "--workers", "8",
    ]
    print("[varroa] Starting full cross-YOLO overnight suite (18 runs, auto resume and HF upload)", flush=True)
    _VarCYSubprocess.run(
        _VarCYCommand,
        cwd=str(_VarCYRoot),
        env={**_VarCYOS.environ, "PYTHONUNBUFFERED": "1"},
        check=True,
    )
    print("[varroa] Finished all cross-YOLO runs, archives and summaries.", flush=True)


In [ ]:
# FTSC JOURNAL PHASE 2 — unattended Run All (no manual phase edits).
# Uses the frozen Phase-1 dataset runner; no additional YOLO portability suite.
# HF failure/provenance mismatch aborts BEFORE authorizing a new training run.
from pathlib import Path as _FTSCPhase2Path
import os as _ftsc_phase2_os
import subprocess as _ftsc_phase2_subprocess
import sys as _ftsc_phase2_sys

_ftsc_phase2_root = _FTSCPhase2Path(yolo_code_dir).resolve()
_ftsc_phase2_runner = _ftsc_phase2_root / 'utils/ftsc_journal_phase2_one_shot.py'
if not _ftsc_phase2_runner.is_file():
    raise FileNotFoundError(f'Missing synced one-shot paper runner: {_ftsc_phase2_runner}. Push codebase first.')
_ftsc_phase2_data_root = (_FTSCPhase2Path(varroa_data_root)).resolve()
_ftsc_phase2_prepared_root = (_FTSCPhase2Path(varroa_prepared_root)).resolve()
_ftsc_phase2_pretrained = (_FTSCPhase2Path(yolo_code_dir) / "yolov8n.pt").resolve()
if not _ftsc_phase2_data_root.is_dir():
    raise FileNotFoundError(f'Dataset not extracted before Phase 2: {_ftsc_phase2_data_root}')
_ftsc_phase2_prepared_root.mkdir(parents=True, exist_ok=True)
_ftsc_phase2_command = [
    _ftsc_phase2_sys.executable, '-u', str(_ftsc_phase2_runner),
    '--dataset', 'varroa',
    '--data-root', str(_ftsc_phase2_data_root),
    '--dataset-root', str(_ftsc_phase2_prepared_root),
    '--pretrained', str(_ftsc_phase2_pretrained),
    '--device', 'cuda',
    '--workers', '8',
]
print('[varroa] FTSC journal one-shot: HF inspect -> verify -> restore/evaluate/resume/train -> publish -> aggregate', flush=True)
_ftsc_phase2_subprocess.run(
    _ftsc_phase2_command,
    cwd=str(_ftsc_phase2_root),
    env={**_ftsc_phase2_os.environ, 'PYTHONUNBUFFERED': '1',
         'PYTHONPATH': str(_ftsc_phase2_root) + _ftsc_phase2_os.pathsep
           + str(_ftsc_phase2_root / 'models_related/ultralytics')},
    check=True,
)
print('[varroa] FTSC journal phase2 finished; see runs/varroa_ftsc_journal_phase2_v1/phase2_progress.json', flush=True)
